##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Autonomous video production with Gemini Omni, video review, and managed agents

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Autonomous_video_production_with_omni_and_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

<!-- Community Contributor Badge -->
<table>
  <tr>
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/Giom-V" target="_blank" title="View Guillaume's profile on GitHub">
        <img src="https://github.com/Giom-V.png?size=100"
             alt="Giom-V's GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/Giom-V" target="_blank"><font color='#217bfe'><strong>Giom</strong></font></a>.</font></h2>
      <h5><font color='black'>Check out Giom's other notebooks <a href="https://github.com/search?q=repo%3Agoogle-gemini%2Fcookbook+%22Giom%22&type=code" target="_blank"><font color="#078efb">here</font></a>.</font></h5><br>
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it too</font></a>!</em></small></font>
    </td>
  </tr>
</table>

Creating high-coherence, extended video sequences with AI requires more than simple text-to-video prompting. Complex choreography, character continuity across shots, and visual defects require an iterative generation and review loop.

In this guide, you will build an autonomous, quality-controlled video production workflow combining five core capabilities:

1. **Gemini Omni Flash (`gemini-omni-1.1-flash`)**: Generate native 10-second video clips with synchronized dialogue and perform multi-turn video extensions.
2. **Dual-mode video understanding (`gemini-3.8-flash`)**: Audit generated footage using both **static multimodal inspection** (broad whole-clip analysis via `previous_interaction_id`) and **agentic video understanding** (frame-by-frame seam boundary inspection, temporal scrubbing, and zoom via Files API upload) returning structured JSON with freeform analysis.
3. **Universal Quality Gates & Physics Verification**: Enforce non-negotiable physical laws on every review: characters must navigate *around* solid obstacles without walking or clipping through rocks/walls, ground contact must be firm, and single-take camera continuity must have strictly zero cuts.
4. **Automated quality gating (EDIT vs REROLL)**: Classify defects into minor localized issues suitable for surgical prompt editing or structural breaks (such as introduced cuts or rock collisions) requiring a complete reroll, followed by regression-proof post-edit verification.
5. **Gemini Agent Skills & Managed Agents (`antigravity-preview-05-2026`)**: Package production tools into an Agent Skill and deploy an autonomous director with separated system instructions and task inputs to cadence a complete 30-second continuous sequence featuring a talkative Mars rover.

## Setup

### Install the SDK

Install the Google GenAI SDK. Version 2.10.0 or higher is required for the Interactions API and Managed Agents.

In [ ]:
%pip install -U -q "google-genai>=2.10.0"

### Set up your API key

Store your Gemini API key in a Colab Secret named `GEMINI_API_KEY`. If you are running outside Colab, set the `GEMINI_API_KEY` environment variable.

In [ ]:
import os

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

### Select models

Select the models for video generation, video review, and the managed agent environment:

In [ ]:
OMNI_MODEL_ID = "gemini-omni-1.1-flash"  # @param ["gemini-omni-1.1-flash", "gemini-omni-flash-preview"] {"allow-input": true, "isTemplate": true}
REVIEW_MODEL_ID = "gemini-3.8-flash"  # @param ["gemini-3.1-pro-preview", "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}
AGENT_ID = "antigravity-preview-05-2026"  # @param ["antigravity-preview-05-2026"] {"allow-input": true, "isTemplate": true}

In [ ]:
# @title Display helpers
import base64
import os
from IPython.display import HTML, display


def show_video_player(video_path_or_bytes, width=640):
    """Displays an HTML5 video player inline."""
    if isinstance(video_path_or_bytes, bytes):
        encoded = base64.b64encode(video_path_or_bytes).decode("ascii")
        data_url = f"data:video/mp4;base64,{encoded}"
    elif os.path.exists(str(video_path_or_bytes)):
        with open(video_path_or_bytes, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("ascii")
        data_url = f"data:video/mp4;base64,{encoded}"
    else:
        data_url = str(video_path_or_bytes)
    html = f'''
    <video width="{width}" controls autoplay muted loop style="border-radius: 8px;">
        <source src="{data_url}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    '''
    display(HTML(html))


def get_output_video(interaction):
    """Retrieve the video part using .output_video or step.content traversal."""
    if hasattr(interaction, "output_video") and interaction.output_video:
        return interaction.output_video
    if hasattr(interaction, "steps") and interaction.steps:
        for step in interaction.steps:
            for item in getattr(step, "content", []):
                if getattr(item, "type", None) == "video":
                    return item
    return None


def get_output_text(interaction):
    """Retrieve the text response using .output_text or step.content traversal."""
    if hasattr(interaction, "output_text") and interaction.output_text:
        return interaction.output_text
    if hasattr(interaction, "steps") and interaction.steps:
        for step in interaction.steps:
            for item in getattr(step, "content", []):
                if getattr(item, "type", None) == "text" and getattr(item, "text", None):
                    return item.text
    return ""


def save_video(interaction, output_path="generated_clip.mp4"):
    """Extracts video bytes from interaction and saves to disk."""
    video = get_output_video(interaction)
    if not video:
        raise ValueError("No video content returned in interaction.")
    if getattr(video, "data", None):
        raw_bytes = base64.b64decode(video.data) if isinstance(video.data, str) else video.data
        with open(output_path, "wb") as f:
            f.write(raw_bytes)
        print(f"Saved video to {output_path} ({len(raw_bytes)} bytes)")
        return output_path
    elif getattr(video, "uri", None):
        print(f"Video URI: {video.uri}")
        return video.uri
    raise ValueError("Video content missing data and uri.")

## Generate a video with Gemini Omni Flash

Gemini Omni Flash (`gemini-omni-1.1-flash`) generates synchronized audio, speech, and video in a single step using the Interactions API. By requesting `response_format={"type": "video"}`, you can direct a continuous 10-second opening shot featuring an expressive talking robot with synchronized dialogue directly from text instructions.

When drafting generation prompts, explicitly define:
- **Physical collision navigation**: Direct the character to steer around solid obstacles (such as boulders and pillars) rather than moving blindly.
- **Timing brackets**: Allocate distinct windows (`[0-3s]`, `[3-8s]`, `[8-10s]`) for movement, dialogue delivery, and resting posture.
- **Word budget**: Keep spoken dialogue within 18–22 words per 10 seconds (~8.0–8.8s) so speech completes cleanly before the clip ends.

In [23]:
turn1_prompt = """
    A continuous cinematic medium shot of an expressive, talking robotic explorer in a temple.
    Sunlight streams through stone arches, illuminating stone floor flagstones and mossy boulders.
    [0-3s] The robot navigates around stone boulders with rhythmic steps and halts in the center.
    [3-8s] The robot faces camera, gestures expressively with its metallic hand, and speaks:
    "Greetings explorer! The planetary alignment has begun. Keep your visual sensors focused."
    [8-10s] The robot finishes speaking, hand settles to its side, maintaining a steady posture.
    Single continuous unbroken camera take, firm ground contact, natural acoustic room tone.
"""

print("Generating Turn 1 video with Gemini Omni Flash...")
turn1 = client.interactions.create(
    model=OMNI_MODEL_ID,
    input=turn1_prompt.strip(),
    response_format={"type": "video"},
)

turn1_path = save_video(turn1, "turn1_talking_robot.mp4")
show_video_player(turn1_path)

Output hidden; open in https://colab.research.google.com to view.

## Review the video with Gemini

In production pipelines, you should never blindly chain video extensions without verifying the preceding clip. Unwanted camera cuts, morphed anatomy, dialogue overlap, or physical collision glitches can ruin subsequent shots.

Because the Interactions API is stateful across turns, you do not need to download and re-upload the video to the Files API for static review. You can directly inspect the generated video by passing `previous_interaction_id=turn1.id` to Gemini.

### Method 1: Static video understanding

In static mode, Gemini analyzes key frames across the clip. By chaining directly from `previous_interaction_id=turn1.id`, Gemini inspects the output video of Turn 1, verifies dialogue delivery, scene stability, physical collision constraints, and returns a structured JSON audit report.

In [24]:
import json

static_audit_prompt = """
    You are an expert film director and technical quality auditor.
    Analyze the video from the previous interaction and return a structured JSON report:

    1. subject_and_dialogue: Is the talking robot visible, delivering dialogue with matched
       gestures?
    2. speech_boundary_hygiene: Does speech conclude cleanly before the final 2 seconds?
    3. general_physics_and_collision: Does the robot obey physical laws? The robot must NOT
       walk through or clip into solid rocks, boulders, walls, or props. Ground contact must be
       firm (no skating, floating, or terrain sinking).
    4. single_take_continuity: Is this a single continuous unbroken camera take with NO cuts,
       camera jumps, or angle snaps?
    5. extension_readiness: Does the video settle in a stable posture suitable for extension?

    Return valid JSON:
    {
      "subject_confirmed": true,
      "speech_and_action_confirmed": true,
      "physics_and_collision_pass": true,
      "rock_clipping_detected": false,
      "single_take_pass": true,
      "speech_boundary_clean": true,
      "visual_quality_score": 9,
      "extension_ready": true,
      "freeform_analysis": "Robot avoids boulders, delivers dialogue, unbroken take.",
      "verdict": "PASS"
    }
"""

static_review = client.interactions.create(
    model=REVIEW_MODEL_ID,
    previous_interaction_id=turn1.id,
    input=static_audit_prompt.strip(),
    response_format={"type": "text", "mime_type": "application/json"},
)

static_result = json.loads(get_output_text(static_review))
print("Static Video Review (JSON):")
print(json.dumps(static_result, indent=2))

Static Video Review (JSON):
{
  "subject_confirmed": true,
  "speech_and_action_confirmed": true,
  "physics_and_collision_pass": true,
  "rock_clipping_detected": false,
  "single_take_pass": true,
  "speech_boundary_clean": true,
  "visual_quality_score": 7,
  "extension_ready": true,
  "freeform_analysis": "Robot navigates clear of boulders, delivers dialogue with synchronized gestures, and maintains a continuous unbroken take.",
  "verdict": "PASS"
}


### Method 2: Agentic video understanding with media processing

When you need temporal precision—such as slowing down the final seconds to ensure posture stability, verifying that feet maintain grounded friction, or zooming into obstacle boundaries—**agentic video understanding** allows Gemini to navigate the video dynamically using temporal search tools.

> **Important: `previous_interaction_id` vs. Agentic Processing**
> In interaction history, previous media is retained with default static processing. To activate
> **agentic video understanding**, upload the saved video to the Files API (`client.files.upload`)
> and pass `processing="agentic"` with the file's URI (`uri=video_file.uri`).
>
> The temporal video agent uses this cloud-stored video to dynamically scrub, slow down, and zoom
> into specific intervals on demand without loading sampled frames into the context window.

In [ ]:
import time

print(f"Uploading {turn1_path} to Files API for agentic processing...")
turn1_file = client.files.upload(file=turn1_path)

while turn1_file.state == "PROCESSING":
    time.sleep(2)
    turn1_file = client.files.get(name=turn1_file.name)

if turn1_file.state == "FAILED":
    raise ValueError(f"Video processing failed: {turn1_file.error}")

print(f"File ready for agentic analysis: {turn1_file.uri}")

agentic_audit_prompt = """
    Perform a forensic timeline inspection of the talking robot clip:
    1. Speech and gesture synchronization: Check whether vocal delivery aligns with the robot's
       hand and head gestures during the dialogue window (3s to 8s).
    2. General physics and collision: Zoom into the robot's feet and surrounding obstacles.
       Verify that the robot does NOT walk through, phase into, or clip into solid boulders,
       pillars, or stone terrain. Ensure firm ground contact with no skating or floating.
    3. Single unbroken take: Scrub the entire timeline to verify NO hidden cuts, splices,
       or angle snaps exist anywhere in the clip.
    4. Audio-visual boundary hygiene (8s to 10s): Slow down the final 2 seconds. Verify that
       speech finishes cleanly without trailing mutters, and robot settles into stable posture.
    5. Terminal posture alignment: Describe the robot's posture at 10.0s for Turn 2 pickup.

    Return valid JSON:
    {
      "speech_sync_confirmed": true,
      "physics_and_collision_pass": true,
      "rock_clipping_detected": false,
      "single_take_pass": true,
      "audio_boundary_clean": true,
      "posture_stable_at_10s": true,
      "freeform_analysis": "Cadence matches gestures, maneuvers around rocks, continuous take.",
      "terminal_posture_description": "Robot standing centered, hand lowered to hip, steady gaze.",
      "verdict": "PASS"
    }
"""

agentic_review = client.interactions.create(
    model=REVIEW_MODEL_ID,
    input=[
        {
            "type": "video",
            "uri": turn1_file.uri,
            "mime_type": turn1_file.mime_type,
            "processing": "agentic",
        },
        {"type": "text", "text": agentic_audit_prompt.strip()},
    ],
    response_format={"type": "text", "mime_type": "application/json"},
)

agentic_result = json.loads(get_output_text(agentic_review))
print("Agentic Forensic Audit (JSON):")
print(json.dumps(agentic_result, indent=2))

print("\n--- Token Telemetry ---")
print(f"Input tokens:  {agentic_review.usage.total_input_tokens}")
print(f"Output tokens: {agentic_review.usage.total_output_tokens}")
for modality in agentic_review.usage.input_tokens_by_modality:
    print(f"  {modality.modality}: {modality.tokens} tokens")

## Extend the video and enforce quality criteria

Gemini Omni allows you to extend existing clips natively by passing `previous_interaction_id=turn1.id`. The model inspects the tail end of Turn 1 and appends a continuous 10-second continuation.

When prompting extensions, ensure you instruct the model to maintain the single continuous take and explicitly navigate around solid obstacles to avoid collision clipping.

Generate Turn 2 as an extension candidate:

In [ ]:
turn2_prompt = """
    Extend this video seamlessly from the exact final frame in a single continuous camera take.
    The robot lowers its hand, maneuvers around a foreground boulder toward an ancient altar,
    and points to a glowing glyph.
    Maintain identical temple architecture, warm sunlight, robot design, and firm ground contact.
    Physical constraint: The robot must navigate around solid obstacles and never clip through rock.
"""

print(f"Extending video from interaction {turn1.id}...")
turn2_candidate = client.interactions.create(
    model=OMNI_MODEL_ID,
    previous_interaction_id=turn1.id,
    input=turn2_prompt.strip(),
    response_format={"type": "video"},
)

turn2_candidate_path = save_video(turn2_candidate, "turn2_candidate.mp4")
show_video_player(turn2_candidate_path)

### Review the extension: Seam boundary inspection and triage

When evaluating an extension, you can leverage two complementary review modes:
- **Static review** via `previous_interaction_id=turn2_candidate.id`: Fast, zero-upload,
  and ideal for verifying overall narrative continuity, prompt compliance, and scene coherence.
- **Agentic review** via Files API upload with `processing="agentic"`: Dynamically scrubs
  the video timeline to inspect the **transition seam (9.5s–11.0s) frame-by-frame**, checking for
  micro-jitters, scale pops, solid geometry collision (e.g. walking through rocks or altar steps),
  and introduced cuts.

Crucially, the review does not simply emit a binary pass/fail; it triages the defect into a
remediation strategy with **precise temporal timestamps**:
- **`"PASS"`**: The footage is clean, physics are respected, and the shot is ready for extension.
- **`"EDIT"`**: The issue is minor and localized (e.g. character hovered in empty air instead of
  physically touching the glyph, slight timing offset) with intact physics and zero cuts. The
  reviewer pinpoints the exact defect window (e.g. `[4.5s - 6.5s]`) and dictates surgical fixes.
- **`"REROLL"`**: The failure is structural (an unwanted cut was introduced, character walked
  through a rock or wall, or anatomy warped) requiring a full regeneration.

In [ ]:
import json
import time

extension_audit_prompt = """
    Perform a forensic timeline inspection of the extended video (Turn 1 into Turn 2):
    1. Seam boundary analysis (9.5s - 11.0s): Inspect the transition frame-by-frame.
       Are there jump cuts, sudden camera jerks, scale jumps, or lighting flickers?
    2. General physics & obstacle collision:
       - Solid geometry collision: Did the robot walk through, phase through, or clip into any
         solid rocks, boulders, walls, or altar architecture? (Solid geometry is impassable).
       - Ground contact: Are feet firmly planted on the floor without floating or skating?
    3. Single-take camera continuity: Confirm that NO cuts, splices, or angle snaps occur anywhere.
    4. Choreography & action check:
       - Did the robot step forward and physically touch the glyph with its index finger, or
         merely point toward the altar from a distance?
       - Did the glyph flare with bright cyan light?
    5. Remediation triage & temporal defect isolation:
       - If criteria met: verdict = "PASS"
       - If minor/localized flaw (e.g. hovered without physical contact): verdict = "EDIT"
         Specify defect_timestamp (e.g. "4.5s - 6.5s"), defect_description, and surgical fix.
       - If severe structural break (jump cut, walking through rock): verdict = "REROLL"

    Return valid JSON:
    {
      "seam_continuity_pass": true,
      "physics_and_collision_pass": true,
      "rock_clipping_detected": false,
      "single_take_pass": true,
      "cut_detected": false,
      "action_pass": false,
      "defect_timestamp": "4.5s - 6.5s",
      "defect_description": "At 5.0s, robot hovered hand in air 20cm away without touching glyph.",
      "surgical_fix_instructions": "Step 20cm closer, press index finger onto glyph at 5s.",
      "freeform_analysis": "Seam is smooth, camera unbroken, but robot hovers without contact.",
      "verdict": "EDIT",
      "issue_severity": "minor"
    }
"""

print(f"Uploading {turn2_candidate_path} for agentic seam inspection...")
turn2_file = client.files.upload(file=turn2_candidate_path)

while turn2_file.state == "PROCESSING":
    time.sleep(2)
    turn2_file = client.files.get(name=turn2_file.name)

if turn2_file.state == "FAILED":
    raise ValueError(f"Video processing failed: {turn2_file.error}")

extension_eval = client.interactions.create(
    model=REVIEW_MODEL_ID,
    input=[
        {
            "type": "video",
            "uri": turn2_file.uri,
            "mime_type": turn2_file.mime_type,
            "processing": "agentic",
        },
        {"type": "text", "text": extension_audit_prompt.strip()},
    ],
    response_format={"type": "text", "mime_type": "application/json"},
)

eval_result = json.loads(get_output_text(extension_eval))
print("Extension Forensic Evaluation (JSON):")
print(json.dumps(eval_result, indent=2))
print("\nKey Metrics:")
print(f"  Verdict:               {eval_result.get('verdict')}")
print(f"  Defect Window:         {eval_result.get('defect_timestamp')}")
print(f"  Defect Description:    {eval_result.get('defect_description')}")
print(f"  Surgical Fix:          {eval_result.get('surgical_fix_instructions')}")
print(f"  Seam Continuity:       {eval_result.get('seam_continuity_pass')}")
print(f"  Physics & Collision:   {eval_result.get('physics_and_collision_pass')}")
print(f"  Rock Clipping:         {eval_result.get('rock_clipping_detected')}")
print(f"  Single Take (No Cuts): {eval_result.get('single_take_pass')}")

### Automated remediation: Surgical EDIT vs REROLL

The audit above verified that seam continuity was flawless and physics were preserved (`rock_clipping_detected: false`), but isolated a localized choreography defect: at 5.0s (`[4.5s - 6.5s]`), the robot pointed toward the altar but hovered in empty air without making direct physical contact with the stone glyph.

Instead of discarding the entire sequence or providing a vague prompt, you can apply a **surgical EDIT**. The edit prompt explicitly references:
1. **The exact timestamp and defect** observed in the flawed attempt.
2. **Clear frame-by-frame corrections** explaining what to change second-by-second (stepping 20cm closer, pressing the index finger firmly onto the glyph's center at 5.0s, and triggering the cyan light flare).
3. **Mandatory physics and continuity rules** (navigating around boulders without clipping, firm ground contact, strictly zero introduced cuts).

In [ ]:
verdict = eval_result.get("verdict", "PASS")

if verdict == "EDIT":
    print("Applying targeted EDIT for localized defect...")
    defect_time = eval_result.get("defect_timestamp", "4.5s - 6.5s")
    defect_desc = eval_result.get(
        "defect_description",
        "At 5.0s, the robot hovered in air pointing at altar without physical contact.",
    )
    fix_guidance = eval_result.get(
        "surgical_fix_instructions",
        "Step 20cm closer, press index finger firmly onto glyph at 5s, flare cyan light at 7s.",
    )

    edit_prompt = f"""
        Extend this video seamlessly from the exact final frame in a single
        continuous camera take.

        DIRECTOR SURGICAL REPAIR [TARGET WINDOW: {defect_time}]:
        - Defect in previous take: {defect_desc}
        - Required surgical fix: {fix_guidance}

        SECOND-BY-SECOND CHOREOGRAPHY & TIMELINE CORRECTIONS:
        [0-4s] The robot lowers its waving hand, steps carefully around the foreground
               mossy boulder on the flagstone floor, and walks up to the stone altar.
        [4-7s] SURGICAL CORRECTION FOR [{defect_time}]:
               Rather than hovering or pointing from a distance, the robot steps 20 cm
               closer to the altar, extends its metallic index finger, and firmly
               presses it directly onto the circular center of the carved stone glyph
               at exactly 5 seconds.
        [7-10s] Upon contact, the touched glyph pulses with brilliant cyan light,
               illuminating the stone runes and casting dynamic blue reflections across
               the robot's chest and arms as it holds its hand steady.

        UNIVERSAL CONSTRAINTS:
        - Strict single unbroken camera take: Absolutely NO cuts, angle snaps, or jumps.
        - Physics & collision: Navigate around boulders. Firm ground contact throughout.
          Strictly zero clipping or phasing through solid stone.
    """
    print(f"Edit Prompt:\n{edit_prompt.strip()}")
    turn2_edited = client.interactions.create(
        model=OMNI_MODEL_ID,
        previous_interaction_id=turn1.id,
        input=edit_prompt.strip(),
        response_format={"type": "video"},
    )
    final_video_path = save_video(turn2_edited, "turn2_edited.mp4")
elif verdict == "REROLL":
    print("Severe defect detected (introduced cut or rock collision). Rerolling Turn 2...")
    turn2_edited = client.interactions.create(
        model=OMNI_MODEL_ID,
        previous_interaction_id=turn1.id,
        input=turn2_prompt.strip(),
        response_format={"type": "video"},
    )
    final_video_path = save_video(turn2_edited, "turn2_rerolled.mp4")
else:
    print("Extension passed quality gates without modifications!")
    turn2_edited = turn2_candidate
    final_video_path = turn2_candidate_path

show_video_player(final_video_path)

### Post-edit review: Enforcing Universal Quality Gates

When requesting a prompt edit to fix a localized issue (such as touching the glyph), generative video models often resolve the requested action but introduce catastrophic side-effects—such as introducing an unwanted camera cut or causing the character to walk directly through a solid rock!

If the post-edit review asks *only* "did it touch the glyph?", it will rubber-stamp defective footage.

To prevent this, **every review must enforce Universal Quality Gates**:
1. **Target Issue Verification**: Did the edit fix the specific requested issue?
2. **General Physics & Collision**: Did the robot walk through, phase through, or clip into solid rocks, boulders, walls, or the altar? (Solid geometry is impassable).
3. **Single-Take Camera Continuity**: Was any cut, angle snap, or camera jump introduced? The entire shot must remain a single unbroken take.

If an edit fixes the target action but introduces a cut or causes the robot to walk through a rock, the review flags a critical regression and triggers a `REROLL`.

In [ ]:
target_issue = eval_result.get(
    "target_problem",
    "The robot pointed toward the altar but did not touch the glyph or trigger a cyan light flare.",
)

focused_check_prompt = f"""
    You are auditing a revised video clip after a directed EDIT.
    Target issue requested to fix:
    "{target_issue}"

    Perform a rigorous two-part inspection:

    PART 1: TARGET ISSUE VERIFICATION
    1. Did the robot physically touch the glyph with its index finger?
    2. Did the glyph flare with bright cyan light?

    PART 2: UNIVERSAL QUALITY GATES & REGRESSION AUDIT (MANDATORY ON ALL REVIEWS)
    3. General Physics & Collision:
       - Did the robot walk through, phase through, or clip into any solid rocks, boulders,
         walls, or altar geometry? (Solid geometry is IMPASSABLE. Phasing through rock = FAIL).
       - Is ground contact physically grounded with plausible weight and no floating or skating?
    4. Single-Take Continuity & Cut Detection:
       - Was ANY cut, splice, angle snap, or camera teleport introduced?
       - The entire video MUST be a single unbroken camera take. Any introduced cut = FAIL.

    Decision Rules:
    - If target issue resolved AND physics pass (no rock clipping) AND single-take passes (no cuts):
      verdict = "PASS"
    - If target issue is resolved BUT a cut was introduced or robot clipped through a rock:
      verdict = "FAIL_NEW_DEFECT" (The edit introduced severe regressions: requires REROLL).
    - If target issue is not resolved:
      verdict = "FAIL_UNRESOLVED"

    Return valid JSON:
    {{
      "specific_issue_resolved": true,
      "physics_and_collision_pass": true,
      "rock_clipping_detected": false,
      "single_take_pass": true,
      "cut_detected": false,
      "freeform_analysis": "Target action verified, no rock clipping, unbroken take.",
      "verdict": "PASS",
      "regression_detected": false,
      "recommendation": "Proceed to next segment or reroll if cut/clipping detected."
    }}
"""

focused_eval = client.interactions.create(
    model=REVIEW_MODEL_ID,
    previous_interaction_id=turn2_edited.id,
    input=focused_check_prompt.strip(),
    response_format={"type": "text", "mime_type": "application/json"},
)

post_edit_result = json.loads(get_output_text(focused_eval))
print("Post-Edit Review Result (JSON):")
print(json.dumps(post_edit_result, indent=2))

print(f"\nIssue Resolved:         {post_edit_result.get('specific_issue_resolved')}")
print(f"Physics & Collision:    {post_edit_result.get('physics_and_collision_pass')}")
print(f"Rock Clipping Detected: {post_edit_result.get('rock_clipping_detected')}")
print(f"Single Take (No Cuts):  {post_edit_result.get('single_take_pass')}")
print(f"Cut Detected:           {post_edit_result.get('cut_detected')}")
print(f"Verdict:                {post_edit_result.get('verdict')}")

is_defective = (
    post_edit_result.get("cut_detected")
    or post_edit_result.get("rock_clipping_detected")
    or post_edit_result.get("verdict") != "PASS"
)

if is_defective:
    print("\n[ALERT] Review detected critical regressions (unwanted cut or rock clipping)!")
    print("Action: Defect cannot be accepted. Trigger REROLL with collision constraints.")
else:
    print("\n[SUCCESS] Edit verified with zero regressions. Ready to extend.")

## Build an agent skill

You are going to build a skill. An **Agent Skill** is a structured, self-contained directory that gives an AI orchestrator specialized domain knowledge, operational procedures, and deterministic tools.

### How an agent skill works and is organized

An agent skill consists of three core components:

1. **`SKILL.md` (Skill manifest)**: The foundational instruction file. It opens with YAML frontmatter specifying `name` and `description`, followed by clear markdown documentation. The documentation instructs how to generate videos, how to extend them seamlessly, and how to review footage.
2. **References (`references/`)**: In-depth operational manuals, rubrics, and checklists that agents can consult on demand without cluttering the primary instruction file. Here, you will provide a dedicated `review_rules.md` detailing Universal Quality Gates, General Physics & Solid Collision rules, Seam Hygiene, and the EDIT vs REROLL decision matrix.
3. **Modular scripts**: Standalone, executable Python scripts that perform discrete operational tasks (`generate_clip.py`, `review_clip.py`, `extend_clip.py`) deterministically. Rather than relying on a rigid monolithic script, the orchestrating agent calls these tools and manages the cadencing itself.

The primary objective of this skill is that it **reviews all generated videos and reruns or edits them automatically** if quality checks fail, instead of blindly generating a video.

Create the `skills/video_director` directory and write the `SKILL.md` manifest:

In [ ]:
import os

skill_directory = "skills/video_director"
os.makedirs(skill_directory, exist_ok=True)

skill_manifest = """---
name: video-director
description: >-
  Autonomous video production, review, and quality-gated extension loops
  using Gemini Omni and Video Understanding.
---

# Video Director Skill

## 1. How to Generate a Video
- Model: `gemini-omni-1.1-flash` via `client.interactions.create()`.
- Set `response_format={"type": "video"}`.
- Always specify 'Single continuous unbroken camera take' to prevent jump cuts.
- Mandate obstacle navigation: 'Steer around boulders and rocks, firm ground contact.'
- Use explicit timing brackets: [0-5s] for movement, [5-10s] for action/settling.
- Enforce acoustic cleanliness: 'Natural room tone, strictly no applause, no cheering.'

## 2. How to Extend a Video
- Pass `previous_interaction_id=parent_id` to chain natively from the exact last frame.
- Describe forward motion and action continuity without repeating static background props.
- Keep camera perspective moving naturally from the predecessor's terminal framing.
- Never introduce cuts or angle snaps across extension turns.

## 3. How to Review a Video
- Model: `gemini-3.8-flash` via `client.interactions.create()`.
- Static mode: Pass `previous_interaction_id=clip_id` directly for fast in-session triage.
- Agentic mode: Upload video via `client.files.upload()` and pass `uri` with
  `processing="agentic"` (or `--mode agentic --video-path path.mp4`) to inspect the timeline.
- Universal Quality Gates (Mandatory on ALL reviews):
  1. General Physics & Collision: Characters/robots must NEVER walk or clip through solid
     rocks, boulders, walls, or props. Ground contact must be firm without skating.
  2. Single-Take Continuity: Strictly NO cuts, angle snaps, or camera jumps.
  3. Audio & Speech: Dialogue must be synchronized and conclude before the final 1.5 seconds.
- Consult `references/review_rules.md` for the complete quality rubric and triage matrix.

## 4. Autonomous Review and Remediation Protocol
- **Core Rule**: Never accept an unreviewed video.
- Every generated clip must be inspected by `review_clip.py`.
- Triage remediation strategy:
  - If verdict is `EDIT`: Reprompt from parent turn with precise micro-timing brackets (only
    when physics and single-take camera continuity are 100% intact).
  - If verdict is `REROLL`: Regenerate from parent turn if an unwanted cut was introduced or
    if the character clipped through solid obstacles.
- After an edit, run a focused review verifying that the target defect was resolved WITHOUT
  introducing new regressions (cuts or rock clipping).
"""

skill_file = os.path.join(skill_directory, "SKILL.md")
with open(skill_file, "w", encoding="utf-8") as f:
    f.write(skill_manifest)

print(f"Wrote skill manifest: {skill_file}")

In [ ]:
references_dir = os.path.join(skill_directory, "references")
os.makedirs(references_dir, exist_ok=True)

review_rules_content = """# Video Quality Review & Remediation Rules

Quality assurance criteria and decision rubrics for multi-turn video production.

## 1. General Physics & Solid Obstacle Collision Rules
Physical integrity is a mandatory universal gate on ALL reviews:
- **Solid Collision Impassability**: Characters, rovers, and props must NEVER walk through,
  phase through, or clip into solid environmental geometry (rocks, boulders, walls, altars).
  Solid obstacles must be physically navigated around. Phasing through rock = CRITICAL FAIL.
- **Firm Ground Contact**: Feet, wheels, or tracks must maintain firm contact with the ground plane.
  Reject floating, skating across terrain, or foot sinking into solid rock.
- **Inertia & Plausibility**: Physical movement must exhibit plausible mass and deceleration.

## 2. Single-Take Camera Continuity & Cut Detection
- **Strict single-take rule**: Every shot and extension must be a continuous single camera take.
- **Zero-cut tolerance**: Strictly reject any cuts, angle snaps, teleports, or spliced frames.
- **Post-edit cut trap**: Video edits often hallucinate an angle switch or cut. Any introduced cut
  in an edited clip is an immediate failure requiring a REROLL.

## 3. Seam Boundary & Transition Hygiene (Turn N -> Turn N+1)
When reviewing chained video extensions:
- **Inspect boundary window (9.5s - 10.5s)**: Examine the seam frame-by-frame.
- **Static review**: Ideal for swift triage and narrative alignment without re-upload.
- **Agentic review**: Uploads the clip to the Files API and passes uri with processing="agentic"
  to dynamically scrub the seam frame-by-frame with temporal search tools.
- **Posture stability**: Verify character geometry and silhouette match seamlessly across boundary.

## 4. Audio & Acoustic Hygiene
- **Silence vacuum rule**: Speech should target 18-22 words per 10s (~8.0s-8.8s).
  Dead air (> 2s) causes autoregressive loop stutters and hallucinated noise.
- **Phantom applause trap**: Strictly reject hallucinated clapping, crowd cheering, or music bleed.
- **Boundary hygiene**: Trailing mutters in final 1-2s of Turn N poison opening of
  Turn N+1. Immediate rejection and reprompt required.

## 5. Remediation Decision Matrix: EDIT vs REROLL
- **PASS**: Meets all criteria (physics pass, zero cuts, clean seam, target action met).
- **EDIT**: Minor localized flaw (timing offset, missing hand gesture) where physics and
  camera continuity are 100% intact. Isolate exact timestamps (e.g. [4.5s - 6.5s]) and provide
  surgical frame-by-frame correction instructions.
- **REROLL**: Any structural break (introduced cut, character walking through a rock or wall,
  warped anatomy, severe camera teleport). Discard and reshoot from parent turn.

## 6. Post-Edit Verification Protocol
Following an EDIT, never execute a narrow review that only checks if the target defect was fixed.
Execute a dual-part review:
1. Did the edit resolve the target defect?
2. Did the edit introduce new regressions (such as an unwanted cut or clipping through a rock)?
If any new defect is introduced, reject the edit and trigger a REROLL.
"""

review_rules_file = os.path.join(references_dir, "review_rules.md")
with open(review_rules_file, "w", encoding="utf-8") as f:
    f.write(review_rules_content)

print(f"Wrote review reference: {review_rules_file}")

### Create modular skill scripts

Create dedicated, single-responsibility scripts that the agent can invoke as tools:

- `generate_clip.py`: Generates an initial 10-second video with Gemini Omni Flash.
- `review_clip.py`: Audits any video interaction using Gemini 3.8 Flash, supporting both full triage (`EDIT` vs `REROLL`) and focused post-edit verification with mandatory Universal Quality Gates (physics, collision, cuts).
- `extend_clip.py`: Chains a 10-second extension from an existing interaction ID.

Notice that there is no monolithic sequence script: the agent itself acts as the director, managing the cadencing, reviewing each clip, deciding whether to edit or reroll, and extending turn by turn.

In [ ]:
generate_script_content = """#!/usr/bin/env python3
\"\"\"Generate an initial video clip with Gemini Omni Flash.\"\"\"

import argparse
import base64
import json
import os
import sys
from google import genai


def save_video_bytes(interaction, output_path):
    if hasattr(interaction, "output_video") and interaction.output_video:
        v = interaction.output_video
        if getattr(v, "data", None):
            raw = base64.b64decode(v.data) if isinstance(v.data, str) else v.data
            with open(output_path, "wb") as f:
                f.write(raw)
            return output_path
    if hasattr(interaction, "steps") and interaction.steps:
        for step in interaction.steps:
            for c in getattr(step, "content", []):
                if getattr(c, "type", None) == "video" and getattr(c, "data", None):
                    raw = base64.b64decode(c.data) if isinstance(c.data, str) else c.data
                    with open(output_path, "wb") as f:
                        f.write(raw)
                    return output_path
    return None


def main():
    parser = argparse.ArgumentParser(description="Generate an initial video clip.")
    parser.add_argument("--prompt", required=True, help="Video prompt")
    parser.add_argument("--output", default="clip.mp4", help="Output video file path")
    parser.add_argument("--model", default="gemini-omni-1.1-flash", help="Model ID")
    args = parser.parse_args()

    api_key = os.environ.get("GEMINI_API_KEY")
    client = genai.Client(api_key=api_key)

    interaction = client.interactions.create(
        model=args.model,
        input=args.prompt,
        response_format={"type": "video"},
    )
    saved = save_video_bytes(interaction, args.output)
    result = {
        "interaction_id": interaction.id,
        "output_path": saved or args.output,
        "status": "SUCCESS",
    }
    print(json.dumps(result))


if __name__ == "__main__":
    main()
"""

review_script_content = """#!/usr/bin/env python3
\"\"\"Review a video clip using Gemini Video Understanding with Universal Quality Gates.\"\"\"

import argparse
import json
import os
import sys
import time
from google import genai


def get_output_text(interaction):
    if hasattr(interaction, "output_text") and interaction.output_text:
        return interaction.output_text
    if hasattr(interaction, "steps") and interaction.steps:
        for step in interaction.steps:
            for c in getattr(step, "content", []):
                if getattr(c, "type", None) == "text" and getattr(c, "text", None):
                    return c.text
    return ""


def main():
    parser = argparse.ArgumentParser(description="Review video from an interaction or file.")
    parser.add_argument("--interaction-id", default="", help="Interaction ID for static review")
    parser.add_argument("--video-path", default="", help="Local video path for agentic review")
    parser.add_argument(
        "--mode",
        choices=["static", "agentic"],
        default="static",
        help="Review mode (static or agentic)",
    )
    parser.add_argument("--criteria", default="", help="Specific criteria to verify")
    parser.add_argument("--focus-issue", default="", help="Target defect to verify after edit")
    parser.add_argument("--model", default="gemini-3.8-flash", help="Model ID")
    args = parser.parse_args()

    api_key = os.environ.get("GEMINI_API_KEY")
    client = genai.Client(api_key=api_key)

    if args.focus_issue:
        prompt = f\"\"\"
        You are an expert director reviewing a video edit made to fix this issue:
        "{args.focus_issue}"

        Perform a strict two-part quality audit:
        1. Target Issue: Was this specific issue successfully resolved?
        2. Universal Quality Gates (Mandatory on ALL reviews):
           - General Physics & Collision: Did the character/robot walk through or clip into
             any solid rocks, boulders, walls, or props? Is ground contact firm?
           - Single-Take Continuity: Was any cut, angle snap, or camera jump introduced?
             The shot must remain a single unbroken camera take.
           - Visual stability: Are there warped anatomy or geometry artifacts?

        Return valid JSON:
        {{
          "specific_issue_resolved": true,
          "physics_and_collision_pass": true,
          "rock_clipping_detected": false,
          "single_take_pass": true,
          "cut_detected": false,
          "freeform_analysis": "Forensic evaluation of target fix, collision, and continuity.",
          "verdict": "PASS",
          "notes": "Observation on fix resolution and regression absence."
        }}
        \"\"\"
    else:
        prompt = f\"\"\"
        Perform a forensic timeline inspection of the video:
        1. General Physics & Collision: Does the character respect solid geometry? The character
           must NOT walk through or clip into solid rocks, boulders, walls, or terrain obstacles.
           Ground contact must be firm and natural (no skating or floating).
        2. Single-Take Camera Continuity: Is the clip a single continuous unbroken camera take
           with NO cuts, angle snaps, or camera teleports?
        3. Seam Boundary (for extensions): Frame-by-frame check at boundary for jitters or pop.
        4. Criteria check: {args.criteria or "Natural motion, dialogue, and scene stability"}.
        5. Triage remediation:
           - "PASS": Criteria met, smooth motion, physics respected, unbroken single take.
           - "EDIT": Minor localized flaw (timing offset) with intact physics and no cuts.
             Specify defect_timestamp (e.g. "4.5s - 6.5s") and surgical fix instructions.
           - "REROLL": Severe flaw: introduced cut, walking through rock, warped geometry.

        Return valid JSON:
        {{
          "physics_and_collision_pass": true,
          "rock_clipping_detected": false,
          "single_take_pass": true,
          "cut_detected": false,
          "seam_continuity_pass": true,
          "action_pass": true,
          "defect_timestamp": "4.5s - 6.5s",
          "defect_description": "Precise defect description if EDIT or REROLL.",
          "surgical_fix_instructions": "Frame-by-frame prompt correction instructions.",
          "freeform_analysis": "Technical analysis of physics, collision, camera, and action.",
          "verdict": "PASS",
          "issue_severity": "none"
        }}
        \"\"\"

    if args.mode == "agentic":
        if not args.video_path:
            print(json.dumps({"error": "--video-path is required for agentic review mode."}))
            sys.exit(1)

        video_file = client.files.upload(file=args.video_path)
        while video_file.state == "PROCESSING":
            time.sleep(2)
            video_file = client.files.get(name=video_file.name)

        if video_file.state == "FAILED":
            print(json.dumps({"error": f"Video processing failed: {video_file.error}"}))
            sys.exit(1)

        review = client.interactions.create(
            model=args.model,
            input=[
                {
                    "type": "video",
                    "uri": video_file.uri,
                    "mime_type": video_file.mime_type,
                    "processing": "agentic",
                },
                {"type": "text", "text": prompt.strip()},
            ],
            response_format={"type": "text", "mime_type": "application/json"},
        )
    elif args.interaction_id:
        review = client.interactions.create(
            model=args.model,
            previous_interaction_id=args.interaction_id,
            input=prompt.strip(),
            response_format={"type": "text", "mime_type": "application/json"},
        )
    else:
        print(json.dumps({"error": "Either --interaction-id or --video-path is required."}))
        sys.exit(1)

    print(get_output_text(review))


if __name__ == "__main__":
    main()
"""

extend_script_content = """#!/usr/bin/env python3
\"\"\"Extend an existing video clip using Gemini Omni Flash.\"\"\"

import argparse
import base64
import json
import os
import sys
from google import genai


def save_video_bytes(interaction, output_path):
    if hasattr(interaction, "output_video") and interaction.output_video:
        v = interaction.output_video
        if getattr(v, "data", None):
            raw = base64.b64decode(v.data) if isinstance(v.data, str) else v.data
            with open(output_path, "wb") as f:
                f.write(raw)
            return output_path
    if hasattr(interaction, "steps") and interaction.steps:
        for step in interaction.steps:
            for c in getattr(step, "content", []):
                if getattr(c, "type", None) == "video" and getattr(c, "data", None):
                    raw = base64.b64decode(c.data) if isinstance(c.data, str) else c.data
                    with open(output_path, "wb") as f:
                        f.write(raw)
                    return output_path
    return None


def main():
    parser = argparse.ArgumentParser(description="Extend a video from previous interaction.")
    parser.add_argument("--previous-id", required=True, help="Previous interaction ID")
    parser.add_argument("--prompt", required=True, help="Extension prompt")
    parser.add_argument("--output", default="extension.mp4", help="Output video file path")
    parser.add_argument("--model", default="gemini-omni-1.1-flash", help="Model ID")
    args = parser.parse_args()

    api_key = os.environ.get("GEMINI_API_KEY")
    client = genai.Client(api_key=api_key)

    interaction = client.interactions.create(
        model=args.model,
        previous_interaction_id=args.previous_id,
        input=args.prompt,
        response_format={"type": "video"},
    )
    saved = save_video_bytes(interaction, args.output)
    result = {
        "interaction_id": interaction.id,
        "output_path": saved or args.output,
        "status": "SUCCESS",
    }
    print(json.dumps(result))


if __name__ == "__main__":
    main()
"""

scripts = {
    "generate_clip.py": generate_script_content,
    "review_clip.py": review_script_content,
    "extend_clip.py": extend_script_content,
}

for script_name, script_content in scripts.items():
    script_path = os.path.join(skill_directory, script_name)
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(script_content)
    print(f"Wrote skill script: {script_path}")

## Launch a managed agent to direct a 30-second sequence

Gemini Managed Agents (`antigravity-preview-05-2026`) operate inside an isolated Linux sandbox. When deploying an autonomous agent to direct video production:

1. **Separate system instruction and task input**: Provide a clear `system_instruction` establishing the director persona, Universal Quality Gates (physics, collision avoidance, single-take continuity), and skill navigation rules, while passing the specific production assignment as `input`.
2. **The official `gemini-skills` repository**: Injects [`google-gemini/gemini-skills`](https://github.com/google-gemini/gemini-skills) into `/.agents/skills/` for up-to-date Gemini API best practices.
3. **Local Video Director Skill**: Mounts `skills/video_director/` containing `SKILL.md`, `references/review_rules.md`, and the modular execution scripts into `/workspace/skills/video_director/`.

### Task assignment: Autonomous cadencing of a 30-second continuous sequence

Instruct the agent to direct a 30-second continuous sequence featuring a **talkative autonomous Mars exploration robot** discovering an ancient alien monolith among rugged Martian craters and rock formations.

The robot is talkative and narrates its scientific telemetry and mission log aloud across each 10-second segment. The agent autonomously manages the cadencing: it generates Segment 1 (0-10s), reviews it with `review_clip.py` enforcing general physics (steering around boulders without clipping solid terrain) and single-take continuity, extends Segment 2 (10-20s), audits the seam transition and obstacle avoidance, extends Segment 3 (20-30s), and audits the final clip, automatically deciding whether to EDIT or REROLL any flawed segment:

In [ ]:
agent_system_instruction = """
    You are an autonomous video director and quality supervisor.
    Your mission is to produce high-coherence, multi-turn video sequences with Gemini Omni.

    Operational Rules:
    1. Always review every generated clip before extending. Never accept uninspected footage.
    2. Mandatory Universal Quality Gates:
       - General Physics & Collision: Characters/robots must NEVER walk or clip through solid rocks,
         walls, or obstacles. Obstacles are physically solid. Ground contact must be firm.
       - Single-Take Continuity: Strictly NO cuts, camera jumps, or angle snaps. The entire 30s
         sequence must remain a single unbroken camera take.
       - Speech & Boundary Hygiene: Dialogue must be synchronized and conclude before the final
         1.5 seconds of each segment.
    3. Consult /.agents/skills/ for official Gemini API conventions and best practices.
    4. Use the modular tools in /workspace/skills/video_director/ (generate_clip.py,
       review_clip.py, extend_clip.py) and adhere to
       /workspace/skills/video_director/references/review_rules.md.
    5. Autonomously cadence the workflow: run review_clip.py after every step, decide whether to
       proceed, apply surgical EDIT, or trigger full REROLL.
"""

agent_task_input = """
    Direct a continuous 30-second video sequence featuring a talkative autonomous Mars exploration
    robot discovering an ancient alien monolith among rugged Martian craters and rock formations.

    The robot is talkative and narrates its mission log aloud in each 10-second segment.

    Cadence the production into 3 seamless 10-second segments with automated review loops:
    1. Segment 1 (0-10s): Generate the opening shot of the talkative robot navigating red Martian
       dunes and rock fields, speaking aloud: "Mission log Sol 342. Navigating Sector Seven. Visual
       sensors detect anomalous geometry ahead."
       Review the clip using review_clip.py. Verify general physics (the robot must steer around
       rocks, NEVER walk or roll through solid boulders or terrain), speech sync, and single-take
       continuity. If defects appear, refine prompt and regenerate until PASS.
    2. Segment 2 (10-20s): Extend from Segment 1. The talkative robot approaches the base of the
       towering alien monolith, speaking: "Approaching structure. Composition is non-terrestrial.
       Activating sub-surface scanner now."
       Review the extension seam frame-by-frame and check physics. If the robot clips into rocks
       or a cut was introduced, trigger REROLL. If minor timing issues, apply surgical EDIT.
    3. Segment 3 (20-30s): Extend from Segment 2. The talkative robot extends an analysis probe,
       scanning glowing alien inscriptions while speaking: "Readings confirmed. The glyphs are
       active. Initiating quantum handshake."
       Review final extension. Ensure no cuts, no collision clipping, and clean concluding speech.

    Provide a final production report summarizing all interaction IDs, review verdicts, and videos.
"""

print(f"Launching managed agent ({AGENT_ID})...")
agent_interaction = client.interactions.create(
    agent=AGENT_ID,
    system_instruction=agent_system_instruction.strip(),
    input=agent_task_input.strip(),
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            },
            {
                "type": "inline",
                "content": skill_manifest,
                "target": "/workspace/skills/video_director/SKILL.md",
            },
            {
                "type": "inline",
                "content": review_rules_content,
                "target": "/workspace/skills/video_director/references/review_rules.md",
            },
            {
                "type": "inline",
                "content": generate_script_content,
                "target": "/workspace/skills/video_director/generate_clip.py",
            },
            {
                "type": "inline",
                "content": review_script_content,
                "target": "/workspace/skills/video_director/review_clip.py",
            },
            {
                "type": "inline",
                "content": extend_script_content,
                "target": "/workspace/skills/video_director/extend_clip.py",
            },
        ],
    },
)

print(f"Agent Status:         {agent_interaction.status}")
print(f"Agent Environment ID: {getattr(agent_interaction, 'environment_id', None)}")

### Inspect the agent trajectory

Examine the autonomous actions taken by the agent in its sandbox environment:

In [ ]:
from IPython.display import Markdown

if hasattr(agent_interaction, "steps") and agent_interaction.steps:
    print(f"Agent completed {len(agent_interaction.steps)} step(s):\n")
    for idx, step in enumerate(agent_interaction.steps):
        step_type = getattr(step, "type", "unknown")
        print(f"Step {idx + 1} [{step_type}]:")
        if hasattr(step, "content") and step.content:
            for part in step.content:
                text = getattr(part, "text", "")
                if text:
                    print(f"  {text[:120]}...")

print("\n--- Agent Report ---")
display(Markdown(get_output_text(agent_interaction)))

## Next steps

In this notebook, you built an autonomous video production pipeline that:

1. Generates 10-second cinematic video clips with **Gemini Omni Flash** featuring expressive spoken dialogue and physical obstacle navigation.
2. Audits clips using both **static multimodal inspection** (direct in-session via `previous_interaction_id`) and **agentic video understanding** (via Files API upload with `processing="agentic"`), returning structured JSON with freeform analysis.
3. Enforces **Universal Quality Gates** across every review: general physics compliance (no walking or clipping through solid rocks/walls, firm ground contact) and single-take continuity (strictly zero cuts).
4. Triages defects into **EDIT** vs **REROLL**, applies targeted prompt repairs with obstacle avoidance, and validates resolutions with regression-proof post-edit reviews.
5. Packages generation, extension, and review rubrics into a reusable **Agent Skill** with dedicated operational references (`references/review_rules.md`).
6. Deploys a **Gemini Managed Agent** with separated system instructions, official `gemini-skills`, and autonomous cadencing to direct a complete 30-second continuous sequence featuring a talkative Mars rover.

### Useful API references

- [Interactions API Overview](https://ai.google.dev/gemini-api/docs/interactions)
- [Gemini Omni Flash Quickstart](../quickstarts/Get_started_Omni.ipynb)
- [Video Understanding Guide](../quickstarts/Video_understanding.ipynb)
- [Managed Agents Quickstart](../quickstarts/Get_started_managed_agents.ipynb)
- [Official Gemini Skills Repository](https://github.com/google-gemini/gemini-skills)

### Related examples

- [Animated Story Video Generation](./Animated_Story_Video_Generation_gemini.ipynb)
- [Analyze a Video: Summarization](./Analyze_a_Video_Summarization.ipynb)